# 9.11 硬件亲和与内核优化 (FlashAttention-3 / FP8)

> 🕐 预估学习时间：40分钟

算子是否吃满 GPU 吞吐，决定训练/推理成本上限。FlashAttention-3 与 FP8（Hopper）是当前硬件亲和的两条主线：减少 HBM 往返、用张量芯低精度矩阵乘。

本节涵盖：
- Roofline：算力 vs 带宽瓶颈
- FlashAttention tiling 直觉与 FA3 改进点
- FP8 缩放（E4M3/E5M2、延迟缩放）
- 融合算子与 CUDA Graph 搭配
- 选型清单


## 1. Roofline：先判断瓶颈

Arithmetic intensity = FLOPs / Bytes。低于拐点 → 带宽绑定（注意力中间矩阵常如此）；高于拐点 → 算力绑定。


In [ ]:
def roofline_bound(flops, bytes_moved, peak_flops=1e15, peak_bw=3e12):
    '''Return time lower bounds from compute and bandwidth.'''
    t_compute = flops / peak_flops
    t_bw = bytes_moved / peak_bw
    return {
        't_compute': t_compute,
        't_bw': t_bw,
        'bound': 'compute' if t_compute > t_bw else 'bandwidth',
        'est_time': max(t_compute, t_bw),
        'intensity': flops / max(bytes_moved, 1),
    }


# Standard attention materializes S=QK^T (n x n)
def attn_costs(n, d, bytes_per=2):
    flops = 2 * n * n * d + 2 * n * n * d  # QK + AV rough
    bytes_moved = (3 * n * d + n * n) * bytes_per  # Q,K,V + S
    return flops, bytes_moved


def flash_costs(n, d, bytes_per=2, block=128):
    # never materialize full S; stream tiles
    flops = 2 * n * n * d + 2 * n * n * d
    bytes_moved = (3 * n * d) * bytes_per * 2  # extra reloads but no n^2
    return flops, bytes_moved


print('=== Roofline: Standard vs Flash Attention ===')
for n in [2048, 8192, 32768]:
    fs, bs = attn_costs(n, 128)
    ff, bf = flash_costs(n, 128)
    rs, rf = roofline_bound(fs, bs), roofline_bound(ff, bf)
    print(f'n={n}: std_bound={rs["bound"]} intensity={rs["intensity"]:.1f} | '
          f'flash_bound={rf["bound"]} intensity={rf["intensity"]:.1f} '
          f'speedup_est={rs["est_time"]/rf["est_time"]:.2f}x')
print(f'\nKey: FlashAttention wins by raising arithmetic intensity (less HBM traffic).')


## 2. FlashAttention tiling 与 FA3 要点

**FA1/2**：分块 softmax，在 SRAM 累积；FA2 改善并行与调度。  
**FA3（Hopper）**：更好利用 TMA/WGMMA、异步流水、FP8 路径，进一步重叠内存与计算。

教学实现用分块在线 softmax 演示数值等价。


In [ ]:
import torch
import math


def online_softmax_attention(Q, K, V, block=32):
    '''Exact attention via tiled online softmax (educational).'''
    B, H, N, D = Q.shape
    out = torch.zeros(B, H, N, D)
    for i0 in range(0, N, block):
        i1 = min(i0 + block, N)
        qi = Q[:, :, i0:i1]
        # running m,l,o
        m = torch.full((B, H, i1 - i0, 1), -1e9)
        l = torch.zeros(B, H, i1 - i0, 1)
        o = torch.zeros(B, H, i1 - i0, D)
        for j0 in range(0, N, block):
            j1 = min(j0 + block, N)
            kj = K[:, :, j0:j1]
            vj = V[:, :, j0:j1]
            s = qi @ kj.transpose(-1, -2) / math.sqrt(D)
            m_new = torch.maximum(m, s.max(dim=-1, keepdim=True).values)
            exp_m = torch.exp(m - m_new)
            p = torch.exp(s - m_new)
            l = l * exp_m + p.sum(dim=-1, keepdim=True)
            o = o * exp_m + p @ vj
            m = m_new
        out[:, :, i0:i1] = o / l
    return out


B, H, N, D = 1, 2, 64, 16
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)
ref = torch.softmax(Q @ K.transpose(-1, -2) / math.sqrt(D), dim=-1) @ V
flash = online_softmax_attention(Q, K, V, block=16)
err = (ref - flash).abs().max().item()
print('=== Tiled Online Softmax ===')
print(f'max abs error vs materialization: {err:.2e}')
print(f'Key: Tiling is mathematically equivalent but keeps n x n scores out of HBM.')



## 3. FP8：格式、缩放与稳定性

| 格式 | 动态范围 | 精度 | 常见用途 |
|------|---------|------|---------|
| E4M3 | 较小 | 较高 | 前向激活/权重 |
| E5M2 | 较大 | 较低 | 梯度 |

需要 **per-tensor / per-block scaling**；延迟缩放（delayed scaling）用历史 amax 选尺度。Hopper FP8 Tensor Core 可接近 2x BF16 吞吐。


In [ ]:
import torch
def fake_fp8_quantize(x, emax=448.0, max_int=448):
    '''Educational absmax scale + fake cast (not bit-exact FP8).'''
    amax = x.detach().abs().amax()
    scale = amax / emax if amax > 0 else torch.tensor(1.0)
    q = torch.clamp((x / scale).round(), -max_int, max_int)
    return q * scale, scale


W = torch.randn(1024, 1024)
A = torch.randn(128, 1024)
Y = A @ W
Wq, sw = fake_fp8_quantize(W)
Aq, sa = fake_fp8_quantize(A)
Yq = Aq @ Wq
rel = (Y - Yq).norm() / Y.norm()
print('=== Fake FP8 Matmul ===')
print(f'rel error={rel.item():.4f} scaleW={sw.item():.3f} scaleA={sa.item():.3f}')

# Delayed scaling: EMA of amax
amax_ema = 0.0
print('delayed amax EMA:')
for t in range(5):
    a = torch.randn(128, 1024) * (1.0 + 0.2 * t)
    amax = a.abs().amax().item()
    amax_ema = 0.9 * amax_ema + 0.1 * amax if t else amax
    print(f'  step={t} amax={amax:.3f} ema={amax_ema:.3f}')
print(f'\nKey: FP8 needs careful scaling; error is tolerable when amax tracked per block/tensor.')



## 4. 融合与图捕获

把 `RMSNorm + QKV + RoPE + Attention` 等串成融合核 / CUDA Graph，减少 launch 与读回。推理小 batch 时常由 CPU launch 主导延迟。


In [ ]:
import torch
import torch.nn.functional as F
import time


def timed(fn, iters=50):
    for _ in range(5):
        fn()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    return (time.perf_counter() - t0) / iters


x = torch.randn(8, 256, 512)
w1 = torch.randn(512, 512)
w2 = torch.randn(512, 512)


def unfused():
    y = x @ w1
    y = F.silu(y)
    y = y @ w2
    return y


# "fused" = fewer Python dispatches (still not a real CUDA fusion)
def fusedish():
    return (F.silu(x @ w1)) @ w2


tu, tf = timed(unfused), timed(fusedish)
print('=== Dispatch Overhead (CPU micro) ===')
print(f'unfused={tu*1e3:.3f}ms fusedish={tf*1e3:.3f}ms')
print(f'Key: Real wins come from CUDA fusion + Graph capture; Python-level fusion is only illustrative.')



## 课后思考题

1. 长序列服务何时从“带宽绑定”转为“算力绑定”？对选型有何影响？
2. FP8 训练中哪些层应保留 BF16？为什么？
3. FA3 与 PagedAttention / PD 分离如何叠加？
4. 如何用 Nsight 验证优化确实提高了 MFU 而非只降了 Python 耗时？

---
> 本节涵盖了9.11 硬件亲和与内核优化的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
